# Chapitre 5 — Attention et Transformers
## Atelier corrigé · M2 IA
NumPy et Matplotlib pour les ateliers 1 et 2 ; PyTorch CPU pour l’atelier 3.
Prédire → vérifier → expliquer.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
print('NumPy :', np.__version__)


## Atelier 1 · Attention masquée
Une ligne entièrement masquée ne doit pas produire de NaN.


In [ ]:
# CELLULE 2 — Attention produit scalaire normalisée, avec masque booléen (True = autorisé).
def masked_softmax(S, mask=None):
    if mask is None:
        mask = np.ones(S.shape, dtype=bool)
    S = np.where(mask, S, -np.inf)
    m = S.max(axis=-1, keepdims=True)
    m = np.where(np.isfinite(m), m, 0.)          # ligne entièrement masquée : pas de NaN
    E = np.where(mask, np.exp(S - m), 0.)
    Z = E.sum(axis=-1, keepdims=True)
    return np.divide(E, Z, out=np.zeros_like(E), where=Z > 0)

def attention(Q, K, V, mask=None):
    """Q : (..., n, d_k) ; K : (..., m, d_k) ; V : (..., m, d_v) ; mask : (..., n, m)."""
    S = Q @ np.swapaxes(K, -1, -2) / np.sqrt(Q.shape[-1])
    A = masked_softmax(S, mask)
    return A @ V, A

def causal_mask(n):
    return np.tril(np.ones((n, n), dtype=bool))

rng = np.random.default_rng(0)
n, d = 6, 8
X = rng.normal(size=(n, d))
Wq, Wk, Wv = (rng.normal(0, d**-.5, (d, d)) for _ in range(3))
out, A = attention(X @ Wq, X @ Wk, X @ Wv, causal_mask(n))
print('Poids causaux (ligne i = requête i) :\n', A)
assert np.allclose(A.sum(-1), 1) and np.allclose(np.triu(A, 1), 0)


### Trois propriétés à vérifier
Prédisez le résultat de chaque test avant de l’exécuter.


In [ ]:
# CELLULE 3 — Trois propriétés à vérifier plutôt qu'à supposer.
def self_attention(X, mask=None):
    return attention(X @ Wq, X @ Wk, X @ Wv, mask)[0]

# (a) Causalité : modifier le jeton 4 ne change pas les sorties 0 à 3.
X2 = X.copy(); X2[4] += 10 * rng.normal(size=d)
diff = np.abs(self_attention(X, causal_mask(n)) - self_attention(X2, causal_mask(n))).max(axis=1)
print('Écart par position après modification du jeton 4 :', diff)
assert np.allclose(diff[:4], 0) and diff[4] > 0

# (b) Sans masque ni position, l'auto-attention est équivariante par permutation.
P = np.eye(n)[rng.permutation(n)]
assert np.allclose(self_attention(P @ X), P @ self_attention(X))

# (c) Le masque causal rompt cette équivariance : l'ordre compte à nouveau.
print('Équivariance avec masque causal :',
      np.allclose(self_attention(P @ X, causal_mask(n)), P @ self_attention(X, causal_mask(n))))

# (d) Masque de padding : les deux dernières positions sont du remplissage.
valid = np.array([1, 1, 1, 1, 0, 0], dtype=bool)
pad_mask = valid[None, :] & valid[:, None]
_, A_pad = attention(X @ Wq, X @ Wk, X @ Wv, pad_mask)
assert np.allclose(A_pad[:, 4:], 0) and np.allclose(A_pad[4:], 0)
print('Lignes de padding (entièrement masquées) :', A_pad[4:].sum(1), '— zéros, pas de NaN')


### L’effet de 1/√d_k
Notez votre prédiction pour l’entropie à d_k = 1024 sans normalisation.


In [ ]:
# CELLULE 4 — Pourquoi diviser par √d_k ? Variance des scores et entropie de l'attention.
def entropy(p):
    return float(-(p * np.log(np.clip(p, 1e-300, None))).sum(-1).mean())

r = np.random.default_rng(1)
dims, m = [4, 16, 64, 256, 1024], 32
rows = []
for dk in dims:
    q = r.normal(size=(500, 1, dk)); k = r.normal(size=(500, m, dk))
    s = (q @ np.swapaxes(k, -1, -2))[:, 0]               # 500 requêtes × 32 clés
    rows.append((dk, s.var(), entropy(masked_softmax(s)), entropy(masked_softmax(s / np.sqrt(dk)))))
for dk, v, h_raw, h_scaled in rows:
    print(f'd_k={dk:5d}  Var(q·k)={v:8.1f}  H sans échelle={h_raw:.2f}  H avec 1/√d_k={h_scaled:.2f}')
print('Entropie maximale log(32) =', np.log(m))
plt.semilogx(dims, [x[2] for x in rows], 'o-', label='sans 1/√d_k')
plt.semilogx(dims, [x[3] for x in rows], 's-', label='avec 1/√d_k')
plt.axhline(np.log(m), ls='--', c='gray', label='uniforme')
plt.xlabel('d_k'); plt.ylabel('Entropie moyenne'); plt.legend(); plt.show()


## Atelier 2 · Bloc Transformer pré-normalisé
Vérifiez le nombre de paramètres avant de calculer quoi que ce soit.


In [ ]:
# CELLULE 5 — Attention multi-têtes et bloc Transformer pré-normalisé (forward NumPy).
def layer_norm(X, g, b, eps=1e-5):
    mu = X.mean(-1, keepdims=True); var = X.var(-1, keepdims=True)
    return g * (X - mu) / np.sqrt(var + eps) + b

def gelu(x):
    return .5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + .044715 * x**3)))

def init_block(d, seed=0):
    r = np.random.default_rng(seed); s = d**-.5
    return {'Wq': r.normal(0, s, (d, d)), 'Wk': r.normal(0, s, (d, d)),
            'Wv': r.normal(0, s, (d, d)), 'Wo': r.normal(0, s, (d, d)),
            'W1': r.normal(0, s, (d, 4*d)), 'c1': np.zeros(4*d),
            'W2': r.normal(0, (4*d)**-.5, (4*d, d)), 'c2': np.zeros(d),
            'g1': np.ones(d), 'b1': np.zeros(d), 'g2': np.ones(d), 'b2': np.zeros(d)}

def split_heads(T, h):          # (B, n, d) -> (B, h, n, d/h)
    B, n, d = T.shape
    return T.reshape(B, n, h, d // h).transpose(0, 2, 1, 3)

def merge_heads(T):             # (B, h, n, d_h) -> (B, n, h·d_h)
    B, h, n, dh = T.shape
    return T.transpose(0, 2, 1, 3).reshape(B, n, h * dh)

def mha(X, P, h, mask=None):
    Q, K, V = (split_heads(X @ P[w], h) for w in ('Wq', 'Wk', 'Wv'))
    O, A = attention(Q, K, V, mask)          # le masque (n, n) est diffusé sur B et h
    return merge_heads(O) @ P['Wo'], A

def block(X, P, h, mask=None):
    X = X + mha(layer_norm(X, P['g1'], P['b1']), P, h, mask)[0]
    H = gelu(layer_norm(X, P['g2'], P['b2']) @ P['W1'] + P['c1'])
    return X + H @ P['W2'] + P['c2']

d, h = 64, 4
P = init_block(d)
n_params = sum(v.size for v in P.values())
print('Paramètres du bloc :', n_params, '; formule 12d² + 9d =', 12*d*d + 9*d)
assert n_params == 12*d*d + 9*d
Xb = np.random.default_rng(2).normal(size=(2, 10, d))
Y = block(Xb, P, h, causal_mask(10))
print('Forme de sortie :', Y.shape)


### Softmax en ligne (principe de FlashAttention)
La preuve est l’exercice 7 du TD.


In [ ]:
# CELLULE 6 — Le principe de FlashAttention : softmax en ligne, par blocs de clés.
def attention_by_blocks(Q, K, V, block_size=4):
    n, dk = Q.shape
    m = np.full((n, 1), -np.inf)        # maximum courant des scores
    l = np.zeros((n, 1))                # somme courante des exponentielles
    O = np.zeros((n, V.shape[1]))       # accumulateur non normalisé
    for start in range(0, K.shape[0], block_size):
        Kb, Vb = K[start:start+block_size], V[start:start+block_size]
        S = Q @ Kb.T / np.sqrt(dk)
        m_new = np.maximum(m, S.max(1, keepdims=True))
        scale = np.exp(m - m_new)       # réajuste ce qui a déjà été accumulé
        E = np.exp(S - m_new)
        l = l * scale + E.sum(1, keepdims=True)
        O = O * scale + E @ Vb
        m = m_new
    return O / l

r = np.random.default_rng(3)
Q, K, V = r.normal(size=(5, 16)), r.normal(size=(23, 16)), r.normal(size=(23, 8))
full, _ = attention(Q, K, V)
for bs in (1, 4, 7, 23):
    assert np.allclose(attention_by_blocks(Q, K, V, bs), full)
print('Attention par blocs = attention complète, sans matrice n × m stockée.')


### Décodage avec cache KV
Pourquoi n’y a-t-il pas de masque dans la version incrémentale ?


In [ ]:
# CELLULE 7 — Décodage avec cache KV : même résultat, beaucoup moins de calcul.
def mha_full_causal(X, P, h):
    return mha(X[None], P, h, causal_mask(len(X)))[0][0]

def mha_incremental(X, P, h):
    cache_k, cache_v, outputs = [], [], []
    for t in range(len(X)):
        x = X[t:t+1][None]                              # un seul nouveau jeton
        q = split_heads(x @ P['Wq'], h)
        cache_k.append(split_heads(x @ P['Wk'], h)); cache_v.append(split_heads(x @ P['Wv'], h))
        K, V = np.concatenate(cache_k, axis=2), np.concatenate(cache_v, axis=2)
        o, _ = attention(q, K, V)                       # pas de masque : le cache ne contient que le passé
        outputs.append(merge_heads(o) @ P['Wo'])
    return np.concatenate(outputs, axis=1)[0]

Xs = np.random.default_rng(4).normal(size=(12, d))
assert np.allclose(mha_full_causal(Xs, P, h), mha_incremental(Xs, P, h))
print('Décodage incrémental avec cache = passage causal complet.')

def kv_cache_bytes(layers, n_kv_heads, d_head, seq, batch=1, bytes_per=2):
    return 2 * layers * n_kv_heads * d_head * seq * batch * bytes_per
for name, kv in [('MHA (32 têtes K/V)', 32), ('GQA (8 groupes)', 8), ('MQA (1 tête K/V)', 1)]:
    print(f'{name:20s} : {kv_cache_bytes(32, kv, 128, 8192) / 2**30:.2f} Gio pour 8 192 jetons')


## Atelier 3 · Un encodeur PyTorch et quatre options de position


In [ ]:
# CELLULE 8 — Un petit encodeur Transformer en PyTorch (CPU), avec quatre options de position.
import math, time
import torch
from torch import nn
import torch.nn.functional as F
torch.set_num_threads(4)

def sinusoidal(n, d):
    pos = torch.arange(n)[:, None]
    freq = torch.exp(-math.log(10000.) * torch.arange(0, d, 2) / d)
    pe = torch.zeros(n, d)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * freq), torch.cos(pos * freq)
    return pe

def rope(x):
    """Rotation des paires de coordonnées : x de forme (B, h, n, d_h)."""
    n, dh = x.shape[-2], x.shape[-1]
    theta = torch.exp(-math.log(10000.) * torch.arange(0, dh, 2) / dh)
    ang = torch.arange(n)[:, None] * theta            # (n, d_h/2)
    cos, sin = ang.cos(), ang.sin()
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack((x1 * cos - x2 * sin, x1 * sin + x2 * cos), dim=-1).flatten(-2)

class MHA(nn.Module):
    def __init__(self, d, h, use_rope=False):
        super().__init__()
        self.h, self.use_rope = h, use_rope
        self.qkv, self.out = nn.Linear(d, 3 * d), nn.Linear(d, d)
    def forward(self, x):
        B, n, d = x.shape
        q, k, v = self.qkv(x).view(B, n, 3, self.h, d // self.h).permute(2, 0, 3, 1, 4)
        if self.use_rope:
            q, k = rope(q), rope(k)
        att = (q @ k.transpose(-1, -2) / math.sqrt(d // self.h)).softmax(-1)
        self.last_attention = att.detach()
        return self.out((att @ v).transpose(1, 2).reshape(B, n, d))

class Block(nn.Module):
    def __init__(self, d, h, use_rope):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = MHA(d, h, use_rope)
        self.ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))          # pré-normalisation
        return x + self.ffn(self.ln2(x))

class Encoder(nn.Module):
    def __init__(self, vocab, n, d=64, h=4, layers=2, position='learned'):
        super().__init__()
        self.position = position
        self.emb = nn.Embedding(vocab, d)
        if position == 'learned':
            self.pos = nn.Parameter(0.02 * torch.randn(n, d))
        elif position == 'sinusoidal':
            self.register_buffer('pos', sinusoidal(n, d))
        self.blocks = nn.ModuleList(Block(d, h, position == 'rope') for _ in range(layers))
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, vocab)
    def forward(self, tokens):
        x = self.emb(tokens)
        if self.position in ('learned', 'sinusoidal'):
            x = x + self.pos[:tokens.shape[1]]
        for b in self.blocks:
            x = b(x)
        return self.head(self.ln(x))

model = Encoder(vocab=10, n=8)
print('Paramètres :', sum(p.numel() for p in model.parameters()))


### Inverser une séquence
Prédisez les quatre accuracies (TD, exercice 3) avant l’entraînement.


In [ ]:
# CELLULE 9 — Tâche : inverser une séquence. Sans information de position, est-ce possible ?
VOCAB, N = 10, 8

def batch(size, generator):
    x = torch.randint(0, VOCAB, (size, N), generator=generator)
    return x, x.flip(1)                      # y_i = x_{N-1-i}

def train(position, steps=1500, seed=0):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(100 + seed)
    model = Encoder(VOCAB, N, position=position)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=.01)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: min(1., (t + 1) / 100))   # warmup
    history = []
    for step in range(steps):
        x, y = batch(128, g)
        loss = F.cross_entropy(model(x).reshape(-1, VOCAB), y.reshape(-1))
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        if step % 50 == 0:
            history.append((step, loss.item()))
    model.eval()
    with torch.no_grad():
        x, y = batch(2000, torch.Generator().manual_seed(999))
        acc = (model(x).argmax(-1) == y).float().mean().item()
    return model, history, acc

results = {}
for position in ['none', 'sinusoidal', 'learned', 'rope']:
    t0 = time.time()
    results[position] = train(position)
    print(f'{position:10s} accuracy par jeton : {results[position][2]:.3f}   ({time.time()-t0:.0f} s)')
print('Hasard :', 1 / VOCAB)


### Courbes et cartes d’attention
Cherchez l’anti-diagonale. Comparez les lignes 1 et 3 du modèle sans position.


In [ ]:
# CELLULE 10 — Courbes d'apprentissage et carte d'attention du modèle appris.
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for position, (_, history, acc) in results.items():
    ax[0].plot(*zip(*history), label=f'{position} ({acc:.0%})')
ax[0].set(xlabel='Mise à jour', ylabel='Entropie croisée', title='Inversion de séquence'); ax[0].legend()
model = results['learned'][0]
with torch.no_grad():
    model(torch.tensor([[3, 1, 4, 1, 5, 9, 2, 6]]))
att = model.blocks[-1].attn.last_attention[0].mean(0)      # moyenne des têtes, dernière couche
ax[1].imshow(att, cmap='Blues'); ax[1].set(xlabel='Clé (position)', ylabel='Requête (position)',
                                          title='Attention, dernière couche')
plt.tight_layout(); plt.show()
